In [1]:
"""
SA Metro GTFS Ingestion Script

Purpose:
- Load SA Metro GTFS feed to AWS S3
- Supports:
    1. Initial full load: Load previous 10 versions
    2. Incremental load: Load the latest version if not already ingested
- Destination S3: <landing_bucket>/gtfs/<FEED_VERSION>
"""

import os
import requests
from io import BytesIO
from pathlib import Path
import yaml
import boto3

# Load AWS credentials from YAML
credentials_file = Path("credentials.yml")

with open(credentials_file, "r") as f:
    credentials = yaml.safe_load(f)

aws_access_key_id = credentials['aws']['aws_access_key_id']
aws_secret_access_key = credentials['aws']['aws_secret_access_key']

# Create AWS session and S3 client
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
s3 = session.client(service_name='s3')

# GTFS Feed Configuration
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_path = "static/latest/version.txt"
feed_url_template = "static/{version}/google_transit.zip"

landing_bucket = "cm-aws-s3-destination"
name = 'phuong'
landing_prefix = f"k1/{name}/gtfs"

#### Initial Full Load: Last 10 Versions

In [2]:
# Get latest GTFS version from API
latest_version_url = f"{base_url}/{latest_version_path}"
response = requests.get(latest_version_url)  
latest_version = int(response.text)
print("Latest GTFS version:", latest_version)

# Determine the 10 previous versions to load
start_version = max(1, latest_version - 10)  
versions_to_load = range(start_version, latest_version)
print(f"Previous 10 versions to load: {list(versions_to_load)}")

# Download & upload to S3
for version in versions_to_load:
    gtfs_url = f"{base_url}/{feed_url_template.format(version=version)}"
    print(f"Processing version {version} from {gtfs_url}")
    
    r = requests.get(gtfs_url)
    if r.status_code != 200:
        print(f"Version {version} not available, skipping")
        continue
    
    # Upload to S3
    s3_key = f"{landing_prefix}/{version}/{version}_google_transit.zip"
    s3.upload_fileobj(Fileobj=BytesIO(r.content), Bucket=landing_bucket, Key=s3_key)      
    print(f"Uploaded version {version} to s3://{landing_bucket}/{s3_key}")

print("All available previous 10 GTFS versions loaded successfully")

Latest GTFS version: 1621
Previous 10 versions to load: [1611, 1612, 1613, 1614, 1615, 1616, 1617, 1618, 1619, 1620]
Processing version 1611 from http://gtfs.adelaidemetro.com.au/v1/static/1611/google_transit.zip
Uploaded version 1611 to s3://cm-aws-s3-destination/k1/phuong/gtfs/1611/1611_google_transit.zip
Processing version 1612 from http://gtfs.adelaidemetro.com.au/v1/static/1612/google_transit.zip
Uploaded version 1612 to s3://cm-aws-s3-destination/k1/phuong/gtfs/1612/1612_google_transit.zip
Processing version 1613 from http://gtfs.adelaidemetro.com.au/v1/static/1613/google_transit.zip
Uploaded version 1613 to s3://cm-aws-s3-destination/k1/phuong/gtfs/1613/1613_google_transit.zip
Processing version 1614 from http://gtfs.adelaidemetro.com.au/v1/static/1614/google_transit.zip
Uploaded version 1614 to s3://cm-aws-s3-destination/k1/phuong/gtfs/1614/1614_google_transit.zip
Processing version 1615 from http://gtfs.adelaidemetro.com.au/v1/static/1615/google_transit.zip
Version 1615 not av

#### Incremental Load: Latest version

In [3]:
def get_api_latest_version():
    """Fetch the latest GTFS version number from the API"""
    url = f"{base_url}/{latest_version_path}"
    response = requests.get(url)
    version = response.text
    return version

def is_version_ingested(version):
    """Check if a specific version already exists in S3"""
    prefix = f"{landing_prefix}/{version}/"
    response = s3.list_objects_v2(Bucket=landing_bucket, Prefix=prefix)
    
    if "Contents" in response:
        print(f"Version {version} already ingested in S3")
        return True
    else:
        print(f"Version {version} not found in S3")
        return False
    
def ingest_version(version):
    """Download GTFS version from API and upload to S3"""
    feed_url = f"{base_url}/static/{version}/google_transit.zip"
    print(f"Downloading GTFS version {version} from {feed_url}")
    
    response = requests.get(feed_url)
    if response.status_code != 200:
        print(f"GTFS version {version} not available. Skipping.")
        return
    
    s3_key = f"{landing_prefix}/{version}/{version}_google_transit.zip"
    s3.upload_fileobj(Fileobj=BytesIO(response.content), Bucket=landing_bucket, Key=s3_key)                 
    print(f"Uploaded version {version} to s3://{landing_bucket}/{s3_key}")

def incremental_load():
    """Check API latest version and ingest if not already in S3"""
    latest_version = get_api_latest_version()
    if not is_version_ingested(latest_version):
        ingest_version(latest_version)
    else:
        print(f"No new version to ingest. Latest version {latest_version} already in S3.")

# Run incremental load
if __name__ == "__main__":
    incremental_load()

Version 1621 not found in S3
Uploaded version 1621 to s3://cm-aws-s3-destination/k1/phuong/gtfs/1621/1621_google_transit.zip
